# C-MAPSS FD001: TinyML, Autoencoder, Hybrid, and AFICv Results

This notebook reuses the versioned local artifacts built from training engines only. The official test labels are loaded only for final scoring.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

from approaches.tinyml import measure_tinyml
from evaluate_test import evaluate_saved_models_on_official_test
from train_models import load_saved_model, train_models

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / 'CMAPSSData'
ARTIFACT_ROOT = PROJECT_ROOT / 'artifacts' / 'FD001'


In [2]:
baseline_runs = train_models(data_dir=DATA_DIR, subset='FD001', reuse=True)
tiny_hybrid_runs = train_models(
    data_dir=DATA_DIR, subset='FD001',
    models=('tinyml_rul', 'hybrid_tinyml_autoencoder_rul'),
    epochs=60, batch_size=256, learning_rate=1e-3, reuse=True,
)
autoencoder_run = train_models(
    data_dir=DATA_DIR, subset='FD001', models=('autoencoder',),
    epochs=30, batch_size=256, hidden_size=32, learning_rate=1e-3, reuse=True,
)
aficv_run = train_models(
    data_dir=DATA_DIR, subset='FD001', models=('random_forest',), aficv_threshold=0.70, reuse=True,
)
autoencoder_metrics = json.loads(autoencoder_run.loc[0, 'validation_metrics'])
print('Autoencoder validation health-signal metrics:')
display(pd.DataFrame([autoencoder_metrics]))


Autoencoder validation health-signal metrics:


,development_early_life_windows,interpretation,spearman_reconstruction_error_vs_rul,validation_reconstruction_mse_mean,validation_reconstruction_mse_median,validation_reconstruction_mse_std,validation_scored_windows
0,1818,Unsupervised health signal only; C-MAPSS has n...,-0.516318,0.412427,0.352082,0.204137,3713


In [3]:
rul_runs = pd.concat([baseline_runs, tiny_hybrid_runs, aficv_run], ignore_index=True)
comparison_metrics, comparison_predictions = evaluate_saved_models_on_official_test(
    rul_runs['artifact_dir'].tolist(), data_dir=DATA_DIR, subset='FD001'
)
aficv_directory = str(aficv_run.loc[0, 'artifact_dir'])
comparison_metrics['model_display'] = comparison_metrics.apply(
    lambda row: 'AFICv Random Forest' if row['artifact_dir'] == aficv_directory else row['model'], axis=1
)
comparison_metrics = comparison_metrics.sort_values('rmse').reset_index(drop=True)
display(comparison_metrics[[
    'model_display', 'samples', 'rmse', 'mae', 'median_absolute_error', 'r2',
    'mape_percent', 'smape_percent', 'bias_predicted_minus_actual',
    'overprediction_count', 'underprediction_count', 'nasa_score', 'nasa_score_mean'
]])
best_rul_model = comparison_metrics.iloc[0]
print(f"Best RUL model by RMSE: {best_rul_model['model_display']} | RMSE={best_rul_model['rmse']:.3f}, MAE={best_rul_model['mae']:.3f}, R2={best_rul_model['r2']:.3f}")
comparison_metrics.to_csv(ARTIFACT_ROOT / 'advanced_model_comparison.csv', index=False)
comparison_predictions.to_csv(ARTIFACT_ROOT / 'advanced_model_predictions.csv', index=False)


,model_display,samples,rmse,mae,median_absolute_error,r2,mape_percent,smape_percent,bias_predicted_minus_actual,overprediction_count,underprediction_count,nasa_score,nasa_score_mean
0,hist_gradient_boosting,100,24.050384,16.884885,11.164060,0.665047,22.209481,20.379879,10.663974,66,34,33604.738351,336.047384
1,random_forest,100,24.141915,16.356815,9.934784,0.662492,21.893343,19.734789,10.379671,75,25,44880.655814,448.806558
2,tinyml_rul,100,24.805804,17.754708,12.045456,0.643674,24.328663,22.153184,10.604829,68,32,7617.857878,76.178579
3,hybrid_tinyml_autoencoder_rul,100,27.316123,19.094873,12.291042,0.567906,27.510433,26.816981,12.432658,69,31,16038.298468,160.382985
4,ridge,100,27.948830,23.104833,21.141298,0.547657,44.483067,46.311433,16.750438,75,25,5434.635517,54.346355
5,AFICv Random Forest,100,31.846360,21.785442,14.647922,0.412700,29.967248,26.274400,10.558162,65,35,252273.192014,2522.731920
6,mean,100,52.795955,40.996113,28.414017,-0.614142,190.837113,56.542391,32.565983,71,29,276878.519370,2768.785194


Best RUL model by RMSE: hist_gradient_boosting | RMSE=24.050, MAE=16.885, R2=0.665


In [4]:
tiny_dir = tiny_hybrid_runs.loc[tiny_hybrid_runs['model'] == 'tinyml_rul', 'artifact_dir'].iloc[0]
tiny_model, _ = load_saved_model(tiny_dir)
tinyml_measurement = pd.DataFrame([measure_tinyml(
    tiny_model.network, np.zeros((1, len(tiny_model.feature_names)), dtype=np.float32), repeats=100
)])
display(tinyml_measurement)
tinyml_measurement.to_csv(ARTIFACT_ROOT / 'tinyml_deployment_measurements.csv', index=False)


C:\Users\devan\Downloads\casestudy1sol\.venv\Lib\site-packages\torch\ao\nn\quantized\modules\utils.py:72: UserWarning: torch.quantize_per_tensor, torch.quantize_per_channel and other quantized tensor creation functions that produce tensors with dtype torch.quint8, torch.qint8, and torch.qint32 are deprecated and will be removed in a future PyTorch release. Please see https://github.com/pytorch/pytorch/issues/184982 for more information. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\quantized\Quantizer.cpp:116.)
  qweight = torch.quantize_per_tensor(


,backend,float_state_dict_bytes,float_latency_ms_per_batch,int8_state_dict_bytes,int8_latency_ms_per_batch,size_reduction_percent,latency_change_percent
0,PyTorch CPU,6357,0.019497,5759,0.317128,9.406953,1526.54767
